# 02 — Entraînement du classifieur

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/SerreIA/blob/main/notebooks/02_entrainement.ipynb)

Un run complet (docs/03-classifieur.md §2) : entraînement suivi dans MLflow, puis
évaluation `test` / `test_xdomain`, planches Grad-CAM et export ONNX vérifié. Sur Colab,
`runs/`, `reports/` et `mlruns/colab/` sont copiés sur le Drive toutes les 10 min ; sur le
PC : `make harvest`.

Sur Colab : *Exécution → Modifier le type d'exécution → GPU (T4)*.

## Paramètres

In [ ]:
CONFIG = "configs/clf_effnet_b0.yaml"
SET = []  # surcharges « cle=valeur », ex. ["train.epochs=5", "augment.background.enabled=false"]
SUBSET = 0  # nb d'images train (0 : tout) ; smoke test : 512
LIMIT_BATCHES = 0  # batches max par epoch (0 : tous) ; smoke test : 5
DEVICE = None  # None : cuda si disponible
GRADCAM_N = 32  # exemples par planche
ONNX_N = 256  # images val pour la vérification ONNX
DATASETS = ["index", "plantvillage", "ccmt", "plantdoc"]

## Environnement

In [ ]:
# Environnement (PC et Colab) : dépôt, données, sorties (docs/data-drive.md).
# PC : rien n'est installé ; un jeu absent est récupéré du Drive par rclone, sinon la cellule
# s'arrête avec la commande à lancer. Colab : clone du dépôt à REV, installation, données via
# le Drive monté (archives MyDrive/SerreIA/data/*.tar), sorties précédentes restaurées.
import logging
import os
import subprocess
import sys
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/SerreIA"  # Colab : data/*.tar et runs/ ; None : pas de Drive
REPO_URL = "https://github.com/lrivals/SerreIA.git"
REV = "main"
COLAB = "google.colab" in sys.modules

if COLAB:
    ROOT = Path("/content/SerreIA")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[vision]"], check=True
    )
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from tools.notebooks import drive_data  # noqa: E402

if COLAB and DRIVE_DIR:
    drive_data.mount_drive()
    drive_data.restore_all(DRIVE_DIR)
drive_data.ensure(DATASETS, DRIVE_DIR if COLAB else None)

# Colab : tous les cœurs pour le chargement, base MLflow séparée de celle du PC.
ENV_SET = (
    [f"data.num_workers={os.cpu_count()}", f"mlflow.tracking_uri={drive_data.COLAB_MLFLOW}"]
    if COLAB
    else []
)

In [ ]:
# Entraînement, évaluation et Grad-CAM tournent sur le GPU, augmentations comprises.
import torch

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
elif COLAB:
    raise RuntimeError("pas de GPU : Exécution → Modifier le type d'exécution → GPU (T4)")
else:
    print("pas de GPU : tout tournera sur CPU (lent)")

In [ ]:
def ensure_backgrounds(cfg):
    """Fonds de serre requis par le remplacement de fond (docs/03 §3.2)."""
    if not cfg["augment"]["background"]["enabled"]:
        return
    try:
        drive_data.ensure(["backgrounds"], DRIVE_DIR if COLAB else None)
    except RuntimeError as e:
        raise RuntimeError(
            f"{e}\nRemplir data/backgrounds/ puis `tools/get_datasets.sh push backgrounds`, "
            'ou SET = ["augment.background.enabled=false"].'
        ) from None

## Configuration

In [ ]:
import yaml

from vision.train.config import apply_overrides, load_config
from vision.train.train_classifier import pick_device, run

cfg = apply_overrides(load_config(ROOT / CONFIG), ENV_SET + SET)
ensure_backgrounds(cfg)
print(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))

## Entraînement

In [ ]:
with drive_data.PeriodicSync(drive_data.OUTPUTS, DRIVE_DIR):
    run_dir = run(cfg, pick_device(DEVICE), SUBSET, LIMIT_BATCHES)
RUN_ID = run_dir.name
print("RUN_ID =", RUN_ID)

## Évaluation in-domain et cross-domain (§4)

In [ ]:
import torch
from IPython.display import Image, Markdown, display

from vision.eval.cross_domain_eval import REPORTS_DIR, log_to_mlflow, run_eval, summary_table

CKPT = ROOT / "runs" / RUN_ID / "best.pt"
device = pick_device(DEVICE)
with torch.inference_mode():
    payload = run_eval(CKPT, device, SUBSET)
OUT = REPORTS_DIR / RUN_ID
log_to_mlflow(RUN_ID, payload, OUT)
display(Markdown(summary_table(payload["results"])))

In [ ]:
for png in sorted(OUT.glob("cm_*_sante.png")):
    print(png.name)
    display(Image(filename=png, width=360))

## Grad-CAM (§5)

In [ ]:
from vision.eval.gradcam import make_sheets

sheets = make_sheets(CKPT, n=GRADCAM_N, save_cams=True, device=device)
for png in sheets:
    if png.name.endswith("_sante.png"):
        print(png.name)
        display(Image(filename=png))

## Export ONNX (§6 : écart < 0,5 point)

In [ ]:
from vision.export.to_onnx import export_and_check

onnx_res = export_and_check(CKPT, ONNX_N)
worst = onnx_res["worst_disagree_pts"]
print(f"écart max {worst:.2f} pt : {'OK' if worst < 0.5 else 'ÉCHEC'}")
drive_data.sync_all(DRIVE_DIR)

## Résumé

Checkpoint : `runs/<RUN_ID>/best.pt` ; rapport : `reports/<RUN_ID>/` (`summary.md`,
matrices, planches) ; MLflow : `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db`
(PC) ou `mlruns/colab/mlflow.db` (récupéré par `make harvest`).